# Single Node Experiment: CPU LLM Inference Scaling with llama.cpp

The first thing we'll have to do is configure our experiment environment. Here, you can choose to either do bare metal or a VM.

In [ ]:
from chi import context

# Auth is preconfigured on jupyterhub. If using externally,
# you can authenticate with a link instead
from chi.jupyterhub import is_jupyterhub_env
if not is_jupyterhub_env():
    context.use_device_auth()

context.use_site("CHI@TACC")
context.choose_project()

# Whether to use baremetal or VM
BAREMETAL = False

In [ ]:
# jupyterhub provides a keypair. If using externally, select one of your existing keypairs
# and set the path to its private key
if not is_jupyterhub_env():
    import os
    import ipywidgets as widgets
    from chi.server import list_keypair
    keypair_dropdown = widgets.Dropdown(options=[k.name for k in list_keypair()], description="Select Keypair")
    keypair_dropdown.observe(lambda change: context.set("keypair_name", change["new"]), names="value")
    context.set("keypair_name", keypair_dropdown.value)
    display(keypair_dropdown)
    context.set("keypair_private_key", os.path.expanduser("~/.ssh/id_rsa"))

## Create a lease

First, we need to reserve the resource we want to use. This will either be a flavor for a VM flavor, or a reservation for a bare metal node type.

Additionally, we'll reserve one floating IP so we can connect to the instance.

In [ ]:
import chi.server
from chi import lease
from datetime import timedelta
import os

my_lease = lease.Lease(f"{os.getenv('USER')}-llm-benchmark", duration=timedelta(hours=2))

if BAREMETAL:
    my_lease.add_node_reservation(node_type="compute_cascadelake_r", amount=1)
else:
    my_flavor = chi.server.get_flavor("m1.xxlarge")
    my_lease.add_flavor_reservation(name=my_flavor.name, amount=1)

my_lease.add_fip_reservation(1)
my_lease.submit(idempotent=True)

## Creating a server

Now, we use our lease and launch a server instance.

In [ ]:
from chi import server

# used for VM
reserved_flavors = my_lease.get_reserved_flavors()
my_flavor_name = reserved_flavors[0].name if reserved_flavors else 'baremetal'

# used for baremetal
reservation_id = my_lease.node_reservations[0]["id"] if my_lease.node_reservations else None

my_server = server.Server(
    f"{os.getenv('USER')}-experiment-instance",
    flavor_name=my_flavor_name,
    image_name="CC-Ubuntu24.04",
    reservation_id=reservation_id,
    key_name=None if is_jupyterhub_env() else context.get("keypair_name"),
)
my_server.submit(idempotent=True)

#### Associating a floating IP address

We can now attach the floating IP to the instance.

In [ ]:
fip = my_server.associate_floating_ip(my_lease.get_reserved_floating_ips()[0])

#### Security Groups
VMs use security groups. These are unique firewall rules that are configurable via OpenStack and the horizon dashboard. This is a convenient way of configuring your VM's security.

By default, all incoming connections to your VM are blocked. In order to allow remote connections, we'll need to assign the "Allow SSH" security group to our VM. We'll find it by listing all the groups below.

In most cases, _this is the ONLY security group you should ever add to your VM._

In [ ]:
if not BAREMETAL:
    from chi import network

    existing_groups = network.list_security_groups(name_filter="ssh")
    if existing_groups:
        sg = existing_groups.pop()
        print(f"Using existing SSH security group - '{sg.name}'")
    else:
        print("Creating new SSH group 'Allow SSH'")
        sg = network.SecurityGroup({"name": "Allow SSH", "description": "Allow incoming SSH connections"})
        sg.add_rule("ingress", "tcp", 22)
        sg.submit()

    my_server.add_security_group(sg.id)

Now, ensure we can reach our instance. For bare metal, it may take a few minutes after the instance becomes active for it to fully become reachable.

In [ ]:
my_server.check_connectivity()

## Configuring the instance

Now that we have a reachable instance, we can upload a setup shell script, and run it.

For this example, we install dependencies, compile llama.cpp, and download a model. On a VM, this may take several minutes.

In [ ]:
my_server.upload("scripts/setup.sh")
my_server.execute("bash setup.sh")

## Running the experiment

We are ready to run the experiment. Similar to configuring the instance, we will upload a benchmark script and execute it.

In this example, the script will run llama-bench for different CPU thread counts. The results will be saved in a CSV file on the remote instance.

In [ ]:
my_server.upload("scripts/benchmark.sh")
my_server.execute("bash benchmark.sh")

## Analyzing results

Since we are in a jupyter environment, it is easy to process the data and analyze it here.

In this example, we'll download the CSV results to this jupyter environment, and then plot them with matplotlib. The code for this is in a seperate file, `bench_results.py` for conveinence. We expect to see that as number of threads increase, prompt processing increases, but token generation, which is memory bottlenecked, will not increase at the same rate.

In [ ]:
from bench_results import download_results, load_results, plot_results

local_dir = download_results(my_server)
df, meta, cpu_model = load_results(local_dir)
plot_results(df, meta, cpu_model, save_path="llm_scaling.png")